In [12]:
import random
import datetime
from faker import Faker
import psycopg2

In [23]:
fake = Faker()
NUM_USERS = 50
NUM_ANONS = 50
ACTIONS_PER_DAY = 5
DAYS = 30

DB_PARAMS = {
    "dbname": "forumdb",
    "user": "user",
    "password": "password",
    "host": "localhost",
    "port": 5432,
}

# Получение ID действия по названию
ACTION_TYPE_IDS = {
    "first_visit": 1,
    "registration": 2,
    "login": 3,
    "logout": 4,
    "topic_create": 5,
    "topic_view": 6,
    "topic_delete": 7,
    "message_post": 8,
}

def connect_db():
    return psycopg2.connect(**DB_PARAMS)

def create_fake_users(conn):
    user_ids = []
    with conn.cursor() as cur:
        for _ in range(100):  # Генерируем 100 пользователей
            username = fake.user_name()
            # Генерируем уникальный email
            email = f"{fake.user_name()}@example.com"
            password = fake.sha256()
            cur.execute("""
                INSERT INTO users (username, email, password_hash, registration_date)
                VALUES (%s, %s, %s, %s) RETURNING user_id
            """, (username, email, password, fake.date_time_this_year()))
            user_ids.append(cur.fetchone()[0])
        conn.commit()
    return user_ids


def create_fake_anons(conn):
    anon_ids = []
    with conn.cursor() as cur:
        for _ in range(NUM_ANONS):
            session_id = fake.uuid4()
            ip = fake.ipv4()
            user_agent = fake.user_agent()
            cur.execute("""
                INSERT INTO anonymous_users (session_id, ip_address, user_agent, first_seen)
                VALUES (%s, %s, %s, %s) RETURNING anon_id
            """, (session_id, ip, user_agent, fake.date_time_this_year()))
            anon_ids.append(cur.fetchone()[0])
        conn.commit()
    return anon_ids

def get_action_id(action_name, conn):
    # Получаем ID действия по имени
    with conn.cursor() as cur:
        cur.execute("""
            SELECT action_type_id 
            FROM action_types 
            WHERE name = %s
        """, (action_name,))
        result = cur.fetchone()
        
        if result:
            return result[0]  # Возвращаем ID действия
        else:
            raise ValueError(f"Действие с именем {action_name} не найдено в базе данных.")


def simulate_logs(conn, user_ids, anon_ids):
    actions = [
        'first_visit', 'registration', 'login', 'logout', 
        'topic_create', 'topic_view', 'topic_delete', 'message_post'
    ]
    
    # Генерация логов за день
    for _ in range(5):  # Генерируем 5 логов для каждого типа действия (можно увеличить для большего числа)
        action = random.choice(actions)  # Случайное действие
        action_id = get_action_id(action, conn)  # Передаем conn в функцию get_action_id
        ts = fake.date_time_this_year()  # Генерация случайной даты
        entity_type = 'topic' if action.startswith('topic') else 'message'
        entity_id = random.randint(1, 100)  # Примерный ID сущности
        response = "success"  # Ответ сервера (например, success)
        additional_info = fake.text()  # Дополнительная информация
        ip = fake.ipv4()  # IP-адрес
        user_agent = fake.user_agent()  # Агент пользователя
        
        # Здесь будет проверка для заполнения правильных полей
        if action == 'registration' or action == 'login':  # Если пользователь залогинился или зарегистрировался
            user_id = random.choice(user_ids)  # Выбираем случайного зарегистрированного пользователя
            anon_id = None
        else:
            user_id = None
            anon_id = random.choice(anon_ids)  # Выбираем случайного анонимного пользователя

        with conn.cursor() as cur:
            # Вставка лога в таблицу
            if user_id:
                cur.execute("""
                    INSERT INTO user_logs (
                        action_type_id, user_id, anon_id, action_time, entity_type, entity_id,
                        server_response, additional_info, ip_address, user_agent
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """, (action_id, user_id, None, ts, entity_type, entity_id, response, additional_info, ip, user_agent))
            elif anon_id:
                cur.execute("""
                    INSERT INTO user_logs (
                        action_type_id, user_id, anon_id, action_time, entity_type, entity_id,
                        server_response, additional_info, ip_address, user_agent
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """, (action_id, None, anon_id, ts, entity_type, entity_id, response, additional_info, ip, user_agent))
            conn.commit()


def main():
    conn = connect_db()
    user_ids = create_fake_users(conn)
    anon_ids = create_fake_anons(conn)
    simulate_logs(conn, user_ids, anon_ids)
    conn.close()
    print("Логи успешно сгенерированы.")

if __name__ == "__main__":
    main()


Логи успешно сгенерированы.
